# Phase 12: Synthetic Stress Test for Relevance Sparsity

## Objective

Provide a small synthetic stress test that demonstrates:
As relevance sparsity increases artificially, failure risk increases.

Earlier phases showed that persistent failures are strongly associated with:
- Extreme relevance sparsity (low num_relevant_1)
- weak score separation

## Method
Using baseline predictions from Phase 6, we:
1. Take existing rankings (scores unchanged)
2. Artificially reduce relevant document counts
3. Recompute Failure@5 under synthetic sparsity
4. Observe whether failure rates increase

This is **NOT** new modeling or causal proof.  
This is a **controlled demonstration** consistent with earlier findings.

## 1. Imports, paths and constants

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings('default')
np.random.seed(36)  #Reproducibility
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

#Paths
PROJECT_ROOT=Path.cwd().parent
PHASE6_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase6_models_folds'
PHASE12_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase12_synthetic_stress'
PHASE12_OUTPUT.mkdir(parents=True, exist_ok=True)

#Constants
DATASET='2007'
FOLD='Fold1'
BASELINE_MODEL='pointwise'
BASELINE_PIPELINE='raw'

#Stress test parameters
SPARSITY_LEVELS=['original', 'cap_rel_3', 'cap_rel_2', 'cap_rel_1']
N_REPETITIONS=50  # Monte Carlo repetitions for averaging
RANDOM_SEED=42

print('='*80)
print('PHASE 12: SYNTHETIC STRESS TEST FOR RELEVANCE SPARSITY')
print('='*80)
print(f'Output: {PHASE12_OUTPUT}')
print(f'Dataset: MQ{DATASET} {FOLD}')
print(f'Baseline: {BASELINE_MODEL}_{BASELINE_PIPELINE}')
print(f'Sparsity levels: {SPARSITY_LEVELS}')
print(f'Repetitions: {N_REPETITIONS}')
print(f'Random seed: {RANDOM_SEED}')
print('\nPurpose: Demonstrate mechanistically that reduced relevance')
print('redundancy increases failure risk in a controlled setting.')
print('='*80)

PHASE 12: SYNTHETIC STRESS TEST FOR RELEVANCE SPARSITY
Output: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase12_synthetic_stress
Dataset: MQ2007 Fold1
Baseline: pointwise_raw
Sparsity levels: ['original', 'cap_rel_3', 'cap_rel_2', 'cap_rel_1']
Repetitions: 50
Random seed: 42

Purpose: Demonstrate mechanistically that reduced relevance
redundancy increases failure risk in a controlled setting.


## 2. Loading Baseline Artifacts

In [3]:
print('\n'+'='*80)
print('LOADING BASELINE ARTIFACTS')
print('='*80)

baseline_key=f'{BASELINE_MODEL}_{BASELINE_PIPELINE}_{DATASET}'
qm_file=PHASE6_OUTPUT / FOLD / f'{baseline_key}_query_metrics.csv'
pred_file=PHASE6_OUTPUT / FOLD / f'{baseline_key}_predictions.csv'

if not qm_file.exists():
    raise RuntimeError(f'Missing: {qm_file}')
if not pred_file.exists():
    raise RuntimeError(f'Missing: {pred_file}')

qm=pd.read_csv(qm_file)
pred=pd.read_csv(pred_file)

#Validating columns
for col in ['qid', 'num_relevant_1', 'Failure@5_primary']:
    if col not in qm.columns:
        raise RuntimeError(f'Missing column in query_metrics: {col}')

for col in ['qid', 'label', 'score']:
    if col not in pred.columns:
        raise RuntimeError(f'Missing column in predictions: {col}')

#Getting evaluable queries
evaluable_qids=set(qm.loc[qm['num_relevant_1']>0, 'qid'])

print(f'Query metrics: {len(qm)} queries')
print(f'Predictions: {len(pred)} documents')
print(f'Evaluable queries: {len(evaluable_qids)}')
print('='*80)


LOADING BASELINE ARTIFACTS
Query metrics: 336 queries
Predictions: 13652 documents
Evaluable queries: 290


## 3. Helper Functions

In [4]:
def cap_relevance(q_docs, max_relevant, rng):
    """
    Cap the number of relevant documents for a single query.
    
    Parameters:
    - q_docs: dataframe of documents for ONE query
    - max_relevant: maximum number of relevant docs to keep
    - rng: numpy random generator for reproducibility
    
    Returns:
    - Modified dataframe with some relevant docs converted to non-relevant
    """
    q_docs=q_docs.copy()
    
    #Identifying relevant docs (label >= 1)
    rel_mask=q_docs['label'] >= 1
    n_relevant=rel_mask.sum()
    
    #If already at or below cap, no change needed
    if n_relevant<=max_relevant:
        return q_docs
    
    #Randomly selecting which relevant docs to keep
    rel_indices=q_docs[rel_mask].index.tolist()
    keep_indices=rng.choice(rel_indices, size=max_relevant, replace=False)
    
    #Converting non-kept relevant docs to non-relevant
    for idx in rel_indices:
        if idx not in keep_indices:
            q_docs.loc[idx, 'label']=0
    
    return q_docs


def compute_failure_at_5(q_docs):
    """
    Compute Failure@5 for a single query's documents.
    
    Returns 1 if failure, 0 if success, np.nan if not evaluable.
    """
    if len(q_docs)==0:
        return np.nan
    
    #Checking evaluability
    n_relevant=(q_docs['label'] >= 1).sum()
    if n_relevant==0:
        return np.nan
    
    #Ranking by score
    q_docs=q_docs.sort_values('score', ascending=False)
    
    #Checking top-5
    k=min(5, len(q_docs))
    top_k=q_docs.head(k)
    
    #Success if at least one relevant in top-k
    has_relevant=(top_k['label'] >= 1).any()
    
    return 0 if has_relevant else 1


print('cap_relevance (FIX 1: single-query input)')
print('compute_failure_at_5')
print('\nHelper functions defined')

cap_relevance (FIX 1: single-query input)
compute_failure_at_5

Helper functions defined


## 4. Running synthetic stress simulation

In [5]:
print('\n'+'='*80)
print('RUNNING SYNTHETIC STRESS SIMULATION')
print('='*80)

#Results storage
results=defaultdict(list)

#Running simulation
for rep in range(N_REPETITIONS):
    if (rep + 1) % 10==0:
        print(f'Repetition {rep + 1}/{N_REPETITIONS}...')
    
    #Creating random generator for this repetition
    rng=np.random.RandomState(RANDOM_SEED + rep)
    
    for level in SPARSITY_LEVELS:
        failures=[]
        
        for qid in evaluable_qids:
            #Getting query predictions
            q_pred=pred[pred['qid'] == qid].copy()
            
            #Applying sparsity cap with corrected signature
            if level=='original':
                q_pred_modified=q_pred
            elif level=='cap_rel_3':
                q_pred_modified=cap_relevance(q_pred, 3, rng)
            elif level=='cap_rel_2':
                q_pred_modified=cap_relevance(q_pred, 2, rng)
            elif level=='cap_rel_1':
                q_pred_modified=cap_relevance(q_pred, 1, rng)
            else:
                raise ValueError(f'Unknown level: {level}')
            
            #Computing failure
            fail=compute_failure_at_5(q_pred_modified)
            
            if not np.isnan(fail):
                failures.append(fail)
        
        #Storing results for this repetition and level
        results[level].append({
            'repetition': rep,
            'n_evaluable': len(failures),
            'n_failures': int(sum(failures)),
            'failure_rate': np.mean(failures) if len(failures) > 0 else np.nan
        })

print(f'\nCompleted {N_REPETITIONS} repetitions across {len(SPARSITY_LEVELS)} levels')
print('='*80)


RUNNING SYNTHETIC STRESS SIMULATION
Repetition 10/50...
Repetition 20/50...
Repetition 30/50...
Repetition 40/50...
Repetition 50/50...

Completed 50 repetitions across 4 levels
